# Chuẩn bị các ô tải Overpass — Hà Nội

Lưới đã tạo có **183 ô 5 × 5 km**, phủ đa giác ranh giới Hà Nội lấy từ OSM tại **01/10/2026 00:00 UTC**. Đây là ô để chia công việc tải; không phải vùng V/N hoặc tuyến khảo sát.

Chọn kernel **Python (PyQGIS 4.2.3)**. Notebook này đọc file và kiểm tra hình học; **không gửi request tải đường**. Ranh giới OSM chưa được đối chiếu với lớp ranh giới pháp lý chính thức.


## 1. Đọc bảng công việc

Mỗi dòng là một ô lá đang hoạt động. `tile_id` nhận diện ô; `side_m` là cạnh tính bằng mét; `scope_fraction` cho biết bao nhiêu phần diện tích ô nằm trong Hà Nội.


In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "work",
              Path("/Users/quanh.hg/python/work")]
WORK_DIR = next((p.resolve() for p in candidates
                 if (p / "data/hanoi_tiles/project_layout.json").is_file()
                 and (p / "scripts/overpass/download_tiles.py").is_file()), None)
if WORK_DIR is None:
    raise FileNotFoundError("Không tìm thấy dự án: cần data/hanoi_tiles và scripts/overpass trong work.")
ROOT = WORK_DIR / "data" / "hanoi_tiles"
SOURCE_DIR = WORK_DIR / "scripts" / "overpass"
loaded = sys.modules.get("download_tiles")
if loaded is not None and (Path(loaded.__file__).resolve().parent != SOURCE_DIR
                           or not hasattr(loaded, "project_file")
                           or Path(loaded.ROOT).resolve() != ROOT):
    raise RuntimeError("Đường dẫn dự án đã đổi. Restart Kernel rồi chạy lại notebook.")
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))
from project_paths import project_file
tiles = pd.read_csv(ROOT / "tiles.csv", keep_default_na=False)
print("Số ô đang hoạt động:", len(tiles))
tiles[["tile_id", "side_m", "scope_fraction", "status"]].head(10)


## 2. Xem bản đồ và kết quả kiểm tra

Bản đồ là lưới ban đầu. Khi đã chia ô, lớp GPKG/GeoJSON phản ánh lưới hiện tại. `uncovered_scope_area_m2` bằng 0 nghĩa là không có diện tích ranh giới bị thiếu ô tải.


In [2]:
from IPython.display import Image, display

qa = json.loads((project_file(ROOT, "qa_report.json")).read_text())
print("Kiểm tra đạt:", qa["passed"])
print("Diện tích ranh OSM (km²):", round(qa["boundary_area_km2"], 2))
print("Diện tích ranh chưa được phủ (m²):", qa["uncovered_scope_area_m2"])
print("Diện tích ô vuông chồng nhau (m²):", qa["projected_square_overlap_area_m2"])
display(Image(filename=str(project_file(ROOT, "hanoi_download_grid.png")), width=800))


## 3. Chọn ô và lấy dữ liệu để gọi API

Ô ví dụ chứa một điểm trung tâm Hà Nội (21.028°N, 105.854°E). Bạn có thể thay mã bằng một mã khác trong bảng.

**BBOX luôn theo thứ tự `(south, west, north, east)`**. Snapshot phải giữ thống nhất giữa các ô. QUERY lấy các ways có `highway`, các restriction liên quan, tín hiệu/barrier và đầy đủ thành viên của đối tượng lấy được.


In [3]:
TILE_ID = "HN_R011_C012"
selected = tiles.loc[tiles["tile_id"] == TILE_ID]
if len(selected) != 1:
    raise ValueError("Mã ô không có trong bảng hiện tại; hãy chọn một ô lá.")
tile = selected.iloc[0]
BBOX = tuple(float(tile[k]) for k in ("south", "west", "north", "east"))
SNAPSHOT_UTC = str(tile["snapshot_utc"])
QUERY = (ROOT / str(tile["query_file"])).read_text(encoding="utf-8")
config = json.loads((ROOT / "tile_project.json").read_text())
ENDPOINT = config["endpoint"]

print("TILE_ID =", repr(TILE_ID))
print("BBOX =", repr(BBOX))
print("SNAPSHOT_UTC =", repr(SNAPSHOT_UTC))
print("Endpoint:", ENDPOINT)
print("\nTruy vấn:\n" + QUERY)


Bạn đã có thông tin để dùng trong `01_overpass_api_download.ipynb`: thay BBOX và snapshot ở cell cấu hình rồi chạy lại từ cell đó trở xuống. Nếu tự gửi request trực tiếp, dùng POST với form `data=QUERY`; giữ bước kiểm tra response và cache của notebook cũ.

Để tải cả Hà Nội, duyệt các dòng trong CSV và lưu từng ô riêng. Bộ tải hàng loạt chưa thuộc notebook này. Ghép kết quả cần khử trùng bằng `(type, id)` trước khi dựng mạng và lọc quyền đi xe máy.


## 4. Kiểm tra lại độ phủ bằng PyQGIS

Hàm kiểm tra ranh giới và lưới hiện tại từ dữ liệu gốc. Nó không gọi API. Ranh giới được chiếu sang EPSG:3405 để tính chiều dài và diện tích bằng đơn vị mét.


In [4]:
sys.path.insert(0, str(SOURCE_DIR))
import build_tiles as grid

report = grid.verify(ROOT)
print(json.dumps(report, ensure_ascii=False, indent=2))


## 5. Chia ô khi đã xác định cần thiết

5 km là kích thước khởi đầu. Có thể chia 5 km → 2,5 km → 1,25 km. Ô cha được thay bằng các ô con có phần diện tích nằm trong Hà Nội; bảng công việc, query và dữ liệu GIS được cập nhật.

Cell dưới **chỉ định nghĩa hàm**, chưa chia ô. Khi thực sự cần, gọi `split_heavy_tile("mã_ô")` trong cell mới. Không chia ô chỉ vì lỗi 429; cần đợi theo phản hồi máy chủ. Xem README về quản lý trạng thái trước khi nối bộ tải hàng loạt.


In [5]:
def split_heavy_tile(tile_id):
    result = grid.split_tile(tile_id, root=ROOT)
    print("Ô con:", result["children"])
    print("Độ phủ sau chia đạt:", result["qa"]["passed"])
    return pd.read_csv(ROOT / "tiles.csv", keep_default_na=False)

# Chỉ gọi khi muốn cập nhật bộ dữ liệu thật:
# tiles = split_heavy_tile("HN_R011_C012")


Mở `hanoi_download_grid.gpkg` trong QGIS để xem ba lớp: `hanoi_boundary` (ranh giới), `download_tiles` (ô tải đầy đủ), `scope_in_tiles` (phần Hà Nội trong mỗi ô). Nguồn: © OpenStreetMap contributors, ODbL, relation 1903516. File PNG là ảnh ban đầu; sau khi chia dùng lớp GIS đã cập nhật.
